In [ ]:
# pip install tensorflow datasets pillow 

In [ ]:
from datasets import load_dataset
import os
import numpy as np
import matplotlib.pyplot as plt
import random
from PIL import Image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

In [ ]:
# loading dataset
dataset = load_dataset("mnist")

In [ ]:
# Function to save images into folders
def save_split(split, split_name):
    for i in range(10):
        os.makedirs(f"data/mnist_data/{split_name}/{i}", exist_ok=True)

    for idx, example in enumerate(split):
        image = example["image"]  # PIL image
        label = example["label"]
        image.save(f"data/mnist_data/{split_name}/{label}/{idx}.png")

In [ ]:
# saving train & test images
save_split(dataset["train"], "train")
save_split(dataset["test"], "test")

In [ ]:
# displaying sample images
fig, axes = plt.subplots(2, 5, figsize=(10, 5))

for i in range(10):
    label = str(i)
    folder = f"data/mnist_data/train/{label}"
    img_file = random.choice(os.listdir(folder))
    
    img = Image.open(os.path.join(folder, img_file))
    
    ax = axes[i // 5, i % 5]
    ax.imshow(img, cmap='gray')
    ax.set_title(f"Label: {label}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# loading data using keras
img_size = 28
batch_size = 32

train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    "data/mnist_data/train",
    target_size=(img_size, img_size),
    color_mode="grayscale",
    batch_size=batch_size,
    class_mode="categorical"
)

test_generator = test_datagen.flow_from_directory(
    "data/mnist_data/test",
    target_size=(img_size, img_size),
    color_mode="grayscale",
    batch_size=batch_size,
    class_mode="categorical"
)

In [ ]:
# defining CNN model
model = models.Sequential([
    layers.Input(shape=(28,28,1)),
    # block 1
    layers.Conv2D(16, (3,3), activation="relu"),
    layers.MaxPooling2D(2,2),
    
    # block 2
    layers.Conv2D(32, (3,3), activation="relu"),
    layers.MaxPooling2D(2,2),

    # block 3
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax")
])

model.summary()

In [ ]:
# compiling the model
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# training the model
history = model.fit(
    train_generator,
    epochs=5,
    validation_data=test_generator
)

In [ ]:
# model evaluation
loss, acc = model.evaluate(test_generator)
print(f"Test Accuracy: {acc:.2f}")

In [ ]:
# making predictions
import numpy as np

# getting one batch
images, labels = next(test_generator)

predictions = model.predict(images)

# showing predictions
fig, axes = plt.subplots(2, 5, figsize=(10,5))

for i in range(10):
    ax = axes[i//5, i%5]
    ax.imshow(images[i].squeeze(), cmap='gray')
    
    pred_label = np.argmax(predictions[i])
    true_label = np.argmax(labels[i])
    
    ax.set_title(f"Pred:{pred_label} True:{true_label}")
    ax.axis("off")

plt.show()

**Exercise** 
1. Create 10 digit images (0–9). Draw them yourself and save as PNG/JPG. Preprocess each image accordingly and run predictions using your trained model. What is the model's performance?
2. Using the Fashion MNIST dataset, load and preprocess the data. Use the same CNN architecture from the MNIST exercise. Train the model for 5 epochs. Report model performance. (Use accuracy and F1 score)

**1**

In [ ]:
# Load and preprocess custom digit images
custom_images_dir = "images"

custom_images = []
true_labels = []

for digit in range(10):
    img_path = os.path.join(custom_images_dir, f"{digit}.png")
    img = Image.open(img_path).convert('L').resize((28, 28))
    img_array = np.array(img) / 255.0
    custom_images.append(img_array)
    true_labels.append(digit)

custom_images_array = np.array(custom_images).reshape(-1, 28, 28, 1)

In [ ]:
# Make predictions
predictions = model.predict(custom_images_array, verbose=0)
predicted_labels = np.argmax(predictions, axis=1)
confidences = np.max(predictions, axis=1) * 100

# Display results
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()

for i in range(10):
    axes[i].imshow(custom_images[i], cmap='gray')
    match = "true" if predicted_labels[i] == true_labels[i] else "False"
    color = 'green' if predicted_labels[i] == true_labels[i] else 'red'
    axes[i].set_title(f"True: {true_labels[i]} | Pred: {predicted_labels[i]} {match}\n{confidences[i]:.1f}%", color=color)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Performance metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns

accuracy = accuracy_score(true_labels, predicted_labels)
precision = precision_score(true_labels, predicted_labels, average='weighted')
recall = recall_score(true_labels, predicted_labels, average='weighted')
f1 = f1_score(true_labels, predicted_labels, average='weighted')

print(f"\nAccuracy:  {accuracy:.2%}")
print(f"Precision: {precision:.2%}")
print(f"Recall:    {recall:.2%}")
print(f"F1-Score:  {f1:.2%}")

# Confusion Matrix
cm = confusion_matrix(true_labels, predicted_labels)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, xticklabels=range(10), yticklabels=range(10))
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.tight_layout()
plt.show()

**2**

In [ ]:
import numpy as np
from sklearn.metrics import f1_score
from tensorflow.keras import layers, models, datasets

# Loading and preprocessing Fashion MNIST
(train_images, train_labels), (test_images, test_labels) = datasets.fashion_mnist.load_data()
train_images = train_images.astype("float32") / 255.0
test_images = test_images.astype("float32") / 255.0
train_images = train_images[..., np.newaxis]
test_images = test_images[..., np.newaxis]

# Build the CNN
fashion_model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(16, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax")
])

# Train and evaluate
fashion_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
fashion_model.fit(train_images, train_labels, epochs=5, verbose=1)

test_loss, test_acc = fashion_model.evaluate(test_images, test_labels, verbose=0)
y_pred = np.argmax(fashion_model.predict(test_images, verbose=0), axis=1)
f1 = f1_score(test_labels, y_pred, average="weighted")

print(f"Test Accuracy: {test_acc:.4f}")
print(f"F1 Score: {f1:.4f}")